# SAMueL-3 data cleaning

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Load data and set up new DataFrame

In [2]:
cleaned_data = pd.DataFrame()
raw_data = pd.read_csv('samuel_3_2020_2025_v2.csv', low_memory=False)
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 501487 entries, 0 to 501486
Data columns (total 89 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   TeamName                                  501487 non-null  str    
 1   AgeUnder40                                501487 non-null  int64  
 2   Age40to44                                 501487 non-null  int64  
 3   Age45to49                                 501487 non-null  int64  
 4   Age50to54                                 501487 non-null  int64  
 5   Age55to59                                 501487 non-null  int64  
 6   Age60to64                                 501487 non-null  int64  
 7   Age65to69                                 501487 non-null  int64  
 8   Age70to74                                 501487 non-null  int64  
 9   Age75to79                                 501487 non-null  int64  
 10  Age80to84                      

## Stroke team

In [3]:
# Strip trailing whitespace
cleaned_data['stroke_team'] = raw_data['TeamName'].str.rstrip()

## Age

In [4]:
# Dictionary defining numeric age as middle of age band
ages: dict = {'AgeUnder40': 37.5,
              'Age40to44': 42.5, 'Age45to49': 47.5,
              'Age50to54': 52.5, 'Age55to59': 57.5,
              'Age60to64': 62.5, 'Age65to69': 67.5,
              'Age70to74': 72.5, 'Age75to79': 77.5,
              'Age80to84': 82.5, 'Age85to89': 87.5,
              'AgeOver90': 92.5}

# Extract age band columns, and find the highest age band that
# # the patient is part of
col_extract: pd.DataFrame = raw_data[ages.keys()]
age_band: pd.Series = col_extract.idxmax(axis=1)

# Use that ageband to find appropriate numeric age from ages dictionary
cleaned_data['age'] = age_band.map(ages)

## Gender

In [5]:
gender: dict = {'M': 1, 'F': 0}
cleaned_data['male'] = raw_data['S01Gender'].map(gender).astype('Int64')

## Onset time

Abbreviations:
* Precise (P)
* Best estimate (BE)
* Not known (NK)
* During sleep (DS)

In [6]:
# Onset to arrival time in minutes
cleaned_data['onset_to_arrival_time'] = raw_data['OnsettoFirstArrivalMinutes'].astype('Int64')

# Whether onset time is known
onset_known: dict = {'NK': 0, 'P': 1, 'BE': 1}
cleaned_data['onset_known'] = raw_data['S01OnsetTimeType'].map(onset_known).astype('Int64')

# Whether onset time is precise - if not, then best estimate or not known
precise_onset_known: dict = {'P': 1, 'BE': 0, 'NK': 0}
cleaned_data['precise_onset_known'] = (
    raw_data['S01OnsetTimeType'].map(precise_onset_known)).astype('Int64')

# Stroke during sleep
sleep: dict = {'DS': 1, 'P': 0, 'BE': 0}
cleaned_data['onset_during_sleep'] = raw_data['S01OnsetDateType'].map(sleep).astype('Int64')

## Ambulance timings

In [7]:
# Arrive by ambulance
by_ambulance: dict = {'Y': 1, 'N': 0}
cleaned_data['arrive_by_ambulance'] = (
    raw_data['S01ArriveByAmbulance'].map(by_ambulance)).astype('Int64')

# Time from call connected to ambulance arrival at patient location
cleaned_data['call_to_ambulance_arrival_time'] = (
    raw_data['CallConnectedtoArrivalMinutes'] -
    raw_data['ArrivalPatientLocationtoArrivalMinutes']).astype('Int64')

# Time that ambulance on scene at patient location
cleaned_data['ambulance_on_scene_time'] = (
        raw_data['ArrivalPatientLocationtoArrivalMinutes'] -
        raw_data['DeparturePatientLocationtoArrivalMinutes']).astype('Int64')

# Ambulance travel time to from patient location to hospital
cleaned_data['ambulance_travel_to_hospital_time'] = (
       raw_data['DeparturePatientLocationtoArrivalMinutes'] -
       raw_data['WheelsStoptoArrivalMinutes']).astype('Int64')

# Ambulance wait time at hospital
cleaned_data['ambulance_wait_time_at_hospital'] = \
    raw_data['WheelsStoptoArrivalMinutes'].astype('Int64')

## Day, month, year and time of arrival.

In [8]:
# Month, year and day
arrival_month_year = pd.to_datetime(
    raw_data['FirstArrivalMonthYear'],
    format='%B %Y'
)

cleaned_data['month'] = arrival_month_year.dt.month
cleaned_data['year'] = arrival_month_year.dt.year
cleaned_data['weekday'] = raw_data['FirstArrivalWeekday']

# Convert weekday to numeric, with Monday=0, Sunday=6
weekday_dict: dict = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
cleaned_data['weekday'] = raw_data['FirstArrivalWeekday'].map(weekday_dict).astype('Int64')

In [9]:
# Get arrival period (3 hour period during day)
arrival_time_dict: dict = {
    '0000to4000': 0,
    '0400to0800': 4,
    '0800to1200': 6,
    '1200to1600': 12,
    '1600to2000': 16,
    '2000to2400': 20
}
cleaned_data['arrival_time_3_hour_period'] = (
    raw_data['FirstArrivalTime'].map(arrival_time_dict).astype('Int64'))

## Stroke type

Abbreviations:
* Infarction (I)
* Primary intracerebral haemorrage (PIH)
* Unknown if not imaged (NaN)

In [10]:
infarction: dict = {'I': 1, 'PIH': 0}
cleaned_data['infarction'] = raw_data['S02StrokeType'].map(infarction)

# LVO (S02LVOInfarction) - Y/N converted to 1/0, with NaN preserved
lvo: dict = {'Y': 1, 'N': 0}
cleaned_data['lvo'] = raw_data['S02LVOInfarction'].map(lvo).astype('Int64')

 ## Scan, thrombolysis and thrombectomy

In [11]:
# Get arrival to scan time
cleaned_data['arrival_to_scan_time'] = raw_data['ArrivaltoBrainImagingMinutes'].astype('Int64')

# Get use of thrombolysis
# NB is the answer automatically selected if type of stroke is PIH
thrombolysis: dict = {'Y': 1, 'N': 0, 'NB': 0, np.nan: 0}
cleaned_data['thrombolysis'] = \
    raw_data['S02Thrombolysis'].map(thrombolysis).astype('Int64')
cleaned_data['thrombolysis_induced_haemorrhage'] = \
    raw_data['S02ThrombolysisCerebralHaemorrhage'].map(thrombolysis).astype('Int64')

# Time from scan to thrombolysis
cleaned_data['scan_to_thrombolysis_time'] = (
    raw_data['ArrivaltoThrombolysisMinutes'] -
    raw_data['ArrivaltoBrainImagingMinutes'])

# Get use of thrombectomy (0 if x is NaN, 1 if x is a number)
cleaned_data['thrombectomy'] = (
    raw_data['ArrivaltoArterialPunctureMinutes'].apply(
        lambda x: 0 if np.isnan(x) else 1))

# Arrival to thrombectomy referral time
cleaned_data['arrival_to_thrombectomy_referral_time'] = (
    raw_data['ArrivaltoIAIReferralMinutes']).astype('Int64')

# AI Aspects performed
cleaned_data['ai_aspects'] = raw_data['S02IAIAspects']

# Aspects score (S02AspectsScore)
cleaned_data['aspects_score'] = raw_data['S02AspectsScore'].astype('Int64')

# Perfusion imaging used
cleaned_data['perfusion_imaging_used'] = (
    raw_data['S02IAIPerfusionImaging'].map(thrombolysis).astype('Int64'))

# Transfer time
cleaned_data['transfer_time'] = \
    raw_data['ArrivaltoRefAmbulanceDepartMinutes'].astype('Int64')

# Time from arrival to thrombectomy
cleaned_data['arrival_to_thrombectomy_time'] = (
    raw_data['ArrivaltoArterialPunctureMinutes']).astype('Int64')

## Comorbidities

These are co-morbidities that were present prior to this admission, and medication that patient was on prior to this admission. The one exception is S2NewAfDiagnosis, which is whether a new diagnosis of atrial fibrillation was made on admission.

In [12]:
comorbidities: dict = {
    'S02CoMCongestiveHeartFailure': 'congestive_heart_failure',
    'S02CoMHypertension': 'hypertension',
    'S02CoMAtrialFibrillation': 'atrial_fibrillation',
    'S02CoMDiabetes': 'diabetes',
    'S02CoMStrokeTIA': 'prior_stroke_tia',
    'S02CoMAFAntiplatelet': 'afib_antiplatelet',
    'S02CoMAFAnticoagulent': 'afib_anticoagulant'}

# Add comorbidites columns with new names and change Y/N/NB to 1/0
cleaned_data[list(comorbidities.values())] = raw_data[comorbidities.keys()]
comorbid_marker = {'Y': 1, 'N': 0, 'NB': 0, np.nan: np.nan}
for col in comorbidities.values():
    cleaned_data[col] = cleaned_data[col].map(comorbid_marker)

# You cannot be marked as receiving antiplatelets unless you have an atrial
# fibrillation diagnosis, so change those from missing to 0
cleaned_data.loc[(cleaned_data['atrial_fibrillation'] == 0) &
                 (cleaned_data['afib_antiplatelet'].isnull()),
                 'afib_antiplatelet'] = 0

In [13]:
anticoag_type: dict = {
    'S02CoMAFAnticoagulentVitK': 'afib_vit_k_anticoagulant',
    'S02CoMAFAnticoagulentDOAC': 'afib_doac_anticoagulant',
    'S02CoMAFAnticoagulentHeparin': 'afib_heparin_anticoagulant'
}

# Add to clean data and map (seperately from other comorbidities as
# dictionary states that leaving empty (NaN) means False for these 3)
cleaned_data[list(anticoag_type.values())] = raw_data[anticoag_type.keys()]
anticoag_type_map = {1: 1, 0: 0, np.nan: 0}
for col in anticoag_type.values():
    cleaned_data[col] = cleaned_data[col].map(anticoag_type_map)

In [14]:
new_afib = {'Y': 1, 'N': 0, np.nan: np.nan}
cleaned_data['new_afib_diagnosis'] = raw_data['S02NewAFDiagnosis'].map(new_afib)

## Prior disability

In [15]:
cleaned_data['prior_disability'] = raw_data['S02RankinBeforeStroke']

## NIHSS data

In [16]:
# Stroke severity is NIHSS score on arrival
cleaned_data['stroke_severity'] = raw_data['S02NihssArrival']

# List of NIHSS arrival measures
nihss: list = ['S02NihssArrivalLoc', 'S02NihssArrivalLocQuestions',
               'S02NihssArrivalLocCommands', 'S02NihssArrivalBestGaze',
               'S02NihssArrivalVisual', 'S02NihssArrivalFacialPalsy',
               'S02NihssArrivalMotorArmLeft', 'S02NihssArrivalMotorArmRight',
               'S02NihssArrivalMotorLegLeft', 'S02NihssArrivalMotorLegRight',
               'S02NihssArrivalLimbAtaxia', 'S02NihssArrivalSensory',
               'S02NihssArrivalBestLanguage', 'S02NihssArrivalDysarthria',
               'S02NihssArrivalExtinctionInattention']

# Finds the minimum value across these columns, and uses that to create
# marker of whether any of them contain a missing value (indicated by -1)
cleaned_data['nihss_complete'] = raw_data[nihss].min(axis=1).apply(
    lambda x: 0 if x == -1 else 1)

# Add columns, setting -1 to NaN
cleaned_data[nihss] = raw_data[nihss]
nihss_nan = [s for s in nihss if s != 'S2NihssArrivalLoc']
cleaned_data[nihss_nan] = cleaned_data[nihss_nan].replace({-1: np.nan})

In [17]:
# Rename NIHSS columns: CamelCase -> snake_case, then remove leading "s02_"
rename_dict = {}
for col in nihss:
    snake = ''.join(['_' + c.lower() if c.isupper() else c for c in col]).lstrip('_')
    cleaned_name = snake.removeprefix('s02_')  # safe: no IndexError if prefix absent
    rename_dict[col] = cleaned_name

cleaned_data.rename(columns=rename_dict, inplace=True)

## Outcome data

In [18]:
# Length of stay in hospital - use ArrivalToHospitalDischargeDays or ArrivalToDeathDays if died in hospital
cleaned_data['length_of_stay'] = (
    raw_data['ArrivalToHospitalDischargeDays'].fillna(raw_data['ArrivalToDeathDays']).astype('Int64'))

cleaned_data['stroke_unit_death'] = (
    raw_data['S07StrokeUnitDeath']
    .map({'Y': 1, 'N': 0})
    .fillna(0)
    .astype('Int64')
)

# Discharge destination
discharge: dict = {
    'CH': 'care_home',
    'D': 'died',
    'H': 'home',
    'SE': 'somewhere_else',
    'TC': 'community_team_or_esd',
    'TCN': 'community_team_or_esd',
    'TN': 'non_ssnap_hospital_team',
    'T': 'ssnap_hospital_team',
}
cleaned_data['discharge_destination'] = (
    raw_data['S07DischargeType'].map(discharge).fillna('missing')
)

# Death - if NaN then 0, if 0+ days (so if died) then 1
cleaned_data['death'] = (raw_data['ArrivalToDeathDays'] >= 0).astype('Int64')

# mRS Outcome
cleaned_data['discharge_disability'] = raw_data['S07RankinDischarge']

# Needs assistance on discharge (S07AdlHelp) 
cleaned_data['needs_assistance_on_discharge'] = (
    raw_data['S07AdlHelp'].map({'Y': 1, 'N': 0, np.nan: np.nan}).astype('Int64')
)

## Deal with unusual results

In [19]:
# Create empty dataframe (with just team) to log issues
issues = pd.DataFrame(cleaned_data['stroke_team'])

**Unusual ambulance times**

In [20]:
amb_col = ['call_to_ambulance_arrival_time',
           'ambulance_on_scene_time',
           'ambulance_travel_to_hospital_time',
           'ambulance_wait_time_at_hospital']

amb_except_wait = ['call_to_ambulance_arrival_time',
                   'ambulance_on_scene_time',
                   'ambulance_travel_to_hospital_time']

# Identify issues

# If any times are missing, set all to NaN
time_all_missing = cleaned_data[amb_col].isnull().all(axis=1)
time_missing = cleaned_data[amb_col].isnull().any(axis=1)

# If any times are negative, set all to NaN
time_negative = (cleaned_data[amb_col] < 0).any(axis=1)

# If any ambulance column except waittime is 0, set all to NaN
time_zero = (cleaned_data[amb_except_wait] == 0).any(axis=1)

# Find individuals with ambulance times yet arrive_by_ambulance = 0
# For those individuals, set ambulance times + arrive by ambulance to NaN
times_but_no_amb = (
    cleaned_data['arrive_by_ambulance'] == 0) & (
        (cleaned_data['call_to_ambulance_arrival_time'].notnull()) |
        (cleaned_data['ambulance_on_scene_time'].notnull()) |
        (cleaned_data['ambulance_travel_to_hospital_time'].notnull()) |
        (cleaned_data['ambulance_wait_time_at_hospital'].notnull()))

# If call to ambulance arrive is greater than 24h (1440m), set all times to NaN
call_time_high = cleaned_data['call_to_ambulance_arrival_time'] > 1440

# If ambulance on scene time is greater than 12h (720m), set all times to NaN
scene_time_high = cleaned_data['ambulance_on_scene_time'] > 720

# If travel to hospital time is greater than 6h (360m), set all times to NaN
travel_time_high = cleaned_data['ambulance_travel_to_hospital_time'] > 360

# If wait time at hospital is greater than 12h (720m), set all times to NaN
wait_time_high = cleaned_data['ambulance_wait_time_at_hospital'] > 720

# Clean the issues
cleaned_data.loc[time_missing, amb_col] = np.nan
cleaned_data.loc[time_negative, amb_col] = np.nan
cleaned_data.loc[time_zero, amb_col] = np.nan
cleaned_data.loc[times_but_no_amb,
                 amb_col + ['arrive_by_ambulance']] = np.nan
cleaned_data.loc[call_time_high, amb_col] = np.nan
cleaned_data.loc[scene_time_high, amb_col] = np.nan
cleaned_data.loc[travel_time_high, amb_col] = np.nan
cleaned_data.loc[wait_time_high, amb_col] = np.nan

# Log the issues
issues['time_some_missing'] = (~time_all_missing & time_missing)
issues['time_negative'] = time_negative
issues['time_zero'] = time_zero
issues['times_but_no_amb'] = times_but_no_amb
issues['call_time_high'] = call_time_high
issues['scene_time_high'] = scene_time_high
issues['travel_time_high'] = travel_time_high
issues['wait_time_high'] = wait_time_high

**Unusual onset to arrival times**

In [21]:
# Identify issues
# If onset to arrival = 0 or is negative, mark as missing
onset_negative = (cleaned_data['onset_to_arrival_time'] <= 0)
# If onset time is not known, set onset to arrival time as missing
onset_unknown = (cleaned_data['onset_known'] == 0)

# Clean the issues
cleaned_data.loc[onset_negative, 'onset_to_arrival_time'] = np.nan
cleaned_data.loc[onset_unknown, 'onset_to_arrival_time'] = np.nan

# Log the issues
issues['onset_negative'] = onset_negative
issues['onset_unknown'] = onset_unknown

**Inconsistent anticoagulant data**

In [22]:
# If discrepancy use of anticoagulants is negative, but an indivudal
# anticoagulant is marked as being used, mark both as missing
anticoag_discrep = (
    (cleaned_data['afib_anticoagulant'] == 0) &
    ((cleaned_data['afib_vit_k_anticoagulant'] == 1) |
     (cleaned_data['afib_doac_anticoagulant'] == 1) |
     (cleaned_data['afib_heparin_anticoagulant'] == 1)))
cleaned_data.loc[
    anticoag_discrep,
    ['afib_anticoagulant', 'afib_vit_k_anticoagulant',
     'afib_doac_anticoagulant', 'afib_heparin_anticoagulant']] = np.nan

# Log the issues
issues['anticoag_discrep'] = anticoag_discrep

**Inconsistent death markers**

In [23]:
# Identify patients who have any marker indicating death
any_death = ((cleaned_data['death'] == 1) |
             (cleaned_data['stroke_unit_death'] == 1) | 
             (cleaned_data['discharge_disability'] == 6) |
             (cleaned_data['discharge_destination'] == 'died'))

# Identify patients who have all markers indicating death
# (or, if not, missing a result)
all_or_missing_death = (
    ((cleaned_data['death'] == 1) |
     cleaned_data['death'].isnull()) &
    ((cleaned_data['stroke_unit_death'] == 1) |
     cleaned_data['stroke_unit_death'].isnull()) &
    ((cleaned_data['discharge_disability'] == 6) |
     (cleaned_data['discharge_disability'].isnull())) &
    ((cleaned_data['discharge_destination'] == 'died') |
     (cleaned_data['discharge_destination'].isnull())))

# Extract columns where have at least one marker indicating death,
# but then others do not (don't extract if its just because others
# are missing) - and set all their death columns as NaN
cleaned_data.loc[
 any_death & ~all_or_missing_death,
 ['death', 'discharge_disability', 'discharge_destination']] = np.nan

# Log issues
issues['inconsistent_death'] = any_death & ~all_or_missing_death

### Remove patients

This is performed at the end of the notebook, as the data cleaning steps assume that the rows in the clean data match up to the rows in the raw data.

In [24]:
# Log issue
issues['missing_stroke_type'] = cleaned_data['infarction'].isna()

# Remove patients missing stroke type
new_cleaned_data = cleaned_data[cleaned_data['infarction'].notna()]

## Save cleaned data

In [25]:
# Save cleaned data
new_cleaned_data.to_csv('cleaned_data.csv', index=False)

# Save issues log
issues.to_csv('data_issues.csv', index=False)